# D_ITEMS - Bronze Layer

This notebook ingests the MIMIC-IV `d_items` source table and stores it as a Bronze Delta table.

A small source check is completed before ingestion to confirm the table structure and key identifier.

In [0]:
from pyspark.sql import functions as F

# Pipeline configuration
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"

SOURCE_PATH = "s3://mimic-iii-demo-bucket/mimic-iv-dataset-demo-2.2-folder/icu/d_items.csv"

# Load source
d_items = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SOURCE_PATH)
)

print(f"Rows: {d_items.count():,}")
print(f"Columns: {len(d_items.columns)}")

d_items.printSchema()

### Source Check

`D_ITEMS` contains the reference information for ICU measurement and event items.

Before creating the Bronze table, `itemid` is checked for missing or duplicate values because it will be used to link this table to ICU event data later.

In [0]:
# Check itemid
total = d_items.count()
unique_items = d_items.select("itemid").distinct().count()
null_items = d_items.filter(F.col("itemid").isNull()).count()

print(f"Rows: {total:,}")
print(f"Unique itemid: {unique_items:,}")
print(f"Null itemid: {null_items:,}")
print(f"Duplicate itemid: {total - unique_items:,}")

### Bronze Ingestion

`itemid` is complete and unique across the source table, so the dataset is ready to be stored in the Bronze layer.

The source structure is preserved and basic ingestion metadata is added for traceability.

In [0]:
# Add ingestion metadata
d_items_bronze = (
    d_items
    .withColumn("_source_file", F.lit("d_items.csv"))
    .withColumn("_source_module", F.lit("icu"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Save as Delta table
(
    d_items_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.d_items_raw")
)

print("Created: workspace.bronze.d_items_raw")

In [0]:
source_count = d_items.count()
bronze_count = spark.table("workspace.bronze.d_items_raw").count()

print(f"Source: {source_count:,}")
print(f"Bronze: {bronze_count:,}")
print(f"Match: {source_count == bronze_count}")

### Bronze Result

`D_ITEMS` was successfully written to the Bronze layer with all source records preserved.

The `itemid` field is complete and unique, and the Bronze table matches the source row count.